In [0]:
# Test timestamp parsing

def test_timestamp_parsing(spark):

    data = [
        ("E001", "2026-09-03T09:00:00", 10.0, 100.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "event_ts", "quantity", "price"]
    )

    result = (
        df.withColumn(
            "event_time",
            F.to_timestamp("event_ts")
        )
    )

    assert result.filter(
        F.col("event_time").isNotNull()
    ).count() == 1


In [0]:
# Test trade value

def test_trade_value(spark):

    data = [
        ("E001", 10.0, 100.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "quantity", "price"]
    )

    result = df.withColumn(
        "trade_value",
        F.col("quantity") * F.col("price")
    )

    row = result.collect()[0]

    assert row.trade_value == 1000.0


In [0]:
# duplicate detection

from pyspark.sql import functions as F

def test_duplicate_detection():
    data = [
        ("E001",),
        ("E001",),
        ("E002",)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id"]
    )

    result = df.dropDuplicates(["event_id"])

    assert result.count() == 2
    print("Duplicate Detection Test Passed")

test_duplicate_detection()


In [0]:
# invalid data

from pyspark.sql import functions as F

valid_condition = (
    F.col("event_id").isNotNull()
    & (F.trim(F.col("event_id")) != "")
    & F.col("side").isin("BUY", "SELL")
    & (F.col("quantity") > 0)
    & (F.col("price") > 0)
)

def test_invalid_side():
    data = [
        ("E001", "HOLD", 10.0, 100.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "side", "quantity", "price"]
    )

    result = df.filter(valid_condition)

    assert result.count() == 0
    print("Invalid Side Test Passed")

test_invalid_side()

In [0]:
# Negative Quantity

def test_negative_quantity():
    data = [
        ("E001", "BUY", -1.0, 100.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "side", "quantity", "price"]
    )

    result = df.filter(valid_condition)

    assert result.count() == 0
    print("Negative Quantity Test Passed")

test_negative_quantity()


In [0]:
# Zero Price

def test_zero_price():
    data = [
        ("E001", "SELL", 10.0, 0.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "side", "quantity", "price"]
    )

    result = df.filter(valid_condition)

    assert result.count() == 0
    print("Zero Price Test Passed")

test_zero_price()


In [0]:
# Valid Record

def test_valid_record():
    data = [
        ("E001", "BUY", 10.0, 100.0)
    ]

    df = spark.createDataFrame(
        data,
        ["event_id", "side", "quantity", "price"]
    )

    result = df.filter(valid_condition)

    assert result.count() == 1
    print("Valid Record Test Passed")

test_valid_record()
